# Demonstration Selection

This Jupyter notebook implements a demonstration selection pipeline for few-shot prompting using the semantic similarity of text samples. The notebook is designed to:

1. Select representative examples from the training datasets using text embeddings and clustering techniques;
2. Create balanced demonstration sets that capture diverse examples for each sentiment class;
3. Utilize a pre-trained embedding model ([Jina V3](https://huggingface.co/jinaai/jina-embeddings-v3)) to capture semantic relationships between text samples;
4. Generate small but representative sets of examples (demos) for each dataset through k-means clustering;
5. Save the selected demonstrations for subsequent use in few-shot prompting evaluations.

The notebook is designed to run locally or in any Jupyter-compatible environment with GPU support and requires environment variables defined in a `.env` file (see `.env.example` at the project root).

<br>

**Expected Environment Variables** (`.env` file)

*   `'AP_DATA_PREPROCESSED'`: Path to preprocessed data


<br>

**Expected project structure**

```
└── Data
   ├── preprocessed
   │   ├── imdb-pt
   │   ├── opcovidbr
   │   ├── reli
   │   └── tweetsentbr
   ├── prompts_templates
   │   ├── template-task-agnostic-no-instructions-few-shot-en.jinja
   │   ├── template-task-specific-instructions-few-shot.jinja
   │   ├── template-task-specific-instructions-few-shot-json.jinja
   │   ├── template-task-specific-instructions-zero-shot.jinja
   │   ├── template-task-specific-meta-prompt-few-shot.jinja
   │   └── template-task-specific-roleplay-few-shot.jinja
   ├── raw
   │   ├── imdb-pt
   │   ├── opcovidbr
   │   ├── reli
   │   └── tweetsentbr
   ├── results
   │   ├── deepseek-v3.2
   │   ├── gemma2-9b-it
   │   ├── qwen3.5-9b
   │   ├── boto-9b-it
   │   └── multilingual-sentiment-analysis
   └── staging_area
```

<br>

**References**:
```
@misc{sturua2024jinaembeddingsv3multilingualembeddingstask,
      title={jina-embeddings-v3: Multilingual Embeddings With Task LoRA},
      author={Saba Sturua and Isabelle Mohr and Mohammad Kalim Akram and Michael Günther and Bo Wang and Markus Krimmel and Feng Wang and Georgios Mastrapas and Andreas Koukounas and Andreas Koukounas and Nan Wang and Han Xiao},
      year={2024},
      eprint={2409.10173},
      archivePrefix={arXiv},
      primaryClass={cs.CL},
      url={https://arxiv.org/abs/2409.10173},
}
```



# Installations

Installing/updating packages.

In [ ]:
!pip install -U transformers

# 00 - Environment Setup

Load environment variables from `.env` file.


In [ ]:
from dotenv import load_dotenv
load_dotenv()

# 01 - Imports

Import the necessary libraries.

In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import random
from transformers import AutoModel
from math import ceil

from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 02 - Constants
Definition of constants.

In [ ]:
#Folder
PREPROCESSED_DATA_ROOT_PATH = os.getenv('AP_DATA_PREPROCESSED')
TRAIN_DATASETS = {Path(dataset).stem.split("_train")[0]:pd.read_csv(dataset) for dataset in glob.glob(f'{PREPROCESSED_DATA_ROOT_PATH}/*/*_train.csv')}

# 03 - Functions

Definition of the functions that will be used during demonstration selection.

In [ ]:
def check_str_or_path(path:str|Path)->bool:
    """
    Check if the given path is a string or a Path object.

    Args:
        path (str | Path): The path to check.

    Returns:
        bool: True if the path is either a string or a Path object, False otherwise.
    """
    if isinstance(path, Path) or isinstance(path, str):
        return True
    else:
       return False

def check_file_exists(path:str|Path)->bool:
    """
    Check if a file exists at the given path.

    Args:
        path (str | Path): The path to the file.

    Returns:
        bool: True if the file exists, False otherwise.
    """
    if check_str_or_path(path):
        return os.path.isfile(path)

def check_directory_exists(path:str|Path)->bool:
    """
    Check if a directory exists at the given path.

    Args:
        path (str | Path): The path to the directory.

    Returns:
        bool: True if the directory exists, False otherwise.
    """
    if check_str_or_path(path):
        return os.path.isdir(path)

def create_directory(path:str|Path)->Path:
    """
    Create a directory at the given path if it does not already exist.

    Args:
        path (str | Path): The path where the directory will be created.

    Returns:
        Path: The Path object of the created directory if successful, or a message indicating that the directory already exists.
    """
    if check_str_or_path(path) and not check_directory_exists(path=path):
        os.mkdir(path)
        return Path(path)
    else:
        return f"Directory {path} already exists"

def generate_embeddings(df: pd.DataFrame, model:AutoModel, task:str="text-matching")->pd.DataFrame:
    """
    Generate text embeddings for a DataFrame column using a given model.
    Args:
        df (pd.DataFrame): The DataFrame containing the text column to encode.
        model (AutoModel): The pre-trained model to use for generating embeddings.
        task (str, optional): The type of encoding task. Defaults to "text-matching".
    Returns:
        pd.DataFrame: The input DataFrame with an additional 'embeddings' column.
    """
    df["embeddings"] = df["text"].apply(lambda x: model.encode(x, task=task))
    return df

def cluster_embeddings(embeddings: pd.Series | np.ndarray | list, n_clusters:int)->tuple:
    """
    Perform K-means clustering on text embeddings.
    Args:
        embeddings (pd.Series | np.ndarray | list): The embeddings to cluster, can be pandas Series, numpy array, or list.
        n_clusters (int): The number of clusters to create.
    Returns:
        tuple: Array of cluster labels assigned to each embedding.
    """
    kmeans = KMeans(n_clusters= n_clusters)
    if isinstance(embeddings, pd.Series):
        embeddings = embeddings.to_list()

    if isinstance(embeddings, list):
        embeddings = np.array(embeddings)

    labels = kmeans.fit_predict(embeddings)
    return labels

def sample_from_clusters(df:pd.DataFrame, random_state:int=42)->pd.DataFrame:
    """
    Sample representative examples from clustered text embeddings for each label.
    Args:
        df (pd.DataFrame): DataFrame containing 'label' and 'embeddings' columns.
    Returns:
        pd.DataFrame: A new DataFrame containing one sample from each cluster for each label.
    """
    classes = df["label"].unique()
    n_clusters = ceil (6/ len(classes))
    samples_list = []

    for label in classes:
        df_label = df[df["label"] == label]
        labels = cluster_embeddings(df_label["embeddings"], n_clusters)
        df_label["cluster_label"] = labels
        for cluster in df_label["cluster_label"].unique():
            df_cluster = df_label[df_label["cluster_label"] == cluster]
            df_cluster = df_cluster.sample(1, random_state=random_state)
            samples_list.append(df_cluster)

    df_sample = pd.concat(samples_list)
    return df_sample

## 04 - Demonstration Selection

## 04.01 - Loading Embedding Model

In [ ]:
model = AutoModel.from_pretrained("jinaai/jina-embeddings-v3", trust_remote_code=True)
model.to("cuda")

## 04.02 - Select Demonstrations for all datasets

In [ ]:
for name, dataset in TRAIN_DATASETS.items():
    file_path = Path(f"{PREPROCESSED_DATA_ROOT_PATH}/{name}")
    file_path.mkdir(parents=True, exist_ok=True)
    df = generate_embeddings(df=dataset, model=model)
    df_sample = sample_from_clusters(df)
    df_sample = df_sample.drop(columns=["embeddings", "cluster_label"])
    df_sample.to_csv(file_path / f"{name}_demos.csv", index=False)